# 🏗️ Notebook 1: Distributed Lock Manager — Requirements & Architecture

## 🛠️ Setup

```bash
cd 06-system-designs/distributed-lock-manager
uv sync
```

Select the `.venv` kernel in VS Code (top-right). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.


## What we're designing

A **distributed lock**: only one client at a time may hold a named lock across a fleet of
services. Used for things like *"only one scheduler runs the nightly job"* or
*"only one worker migrates this customer's data"*.

### Real-world systems you've used that rely on this
- **Google Chubby** — the grandparent; coordinates GFS, Bigtable leader election.
- **Apache ZooKeeper** — powers Kafka controller election, HBase master, Solr leader shards.
- **etcd** — leader election for Kubernetes controllers, CoreDNS, Patroni (Postgres HA).
- **Redis** `SET NX PX` / Redlock — fast cache-coherent locks for app-level jobs.
- **AWS DynamoDB conditional writes** — a cheap lock you can build on a single row.

### Functional requirements
- `acquire(name, owner, ttl)` → success/failure.
- `release(name, owner)` → releases **only if still owner**.
- Locks **expire** (TTL) so a dead owner doesn't hold forever.
- Support **renewal** (heartbeat to extend TTL).
- Return a **fencing token** (monotonic integer) on every successful acquire.

### Non-functional
- **Safety**: at most one *effective* writer at a time, *even under network partitions,
  clock skew, and GC pauses*.
- **Liveness**: if nobody holds the lock, someone can eventually acquire it.
- **Low latency** (1–5 ms local, 10–50 ms cross-region).


## Why is this hard? Let's *see* the problem first.

Before we talk about fencing tokens or Raft, let's watch what happens when multiple
"workers" try to update a shared counter **without** any lock. This is the bug that
a distributed lock is supposed to prevent.


In [ ]:
# BAD -- no lock. Ten threads each try to increment a shared counter 1,000 times.
# Expected: 10,000. We force the race to be visible by yielding between the
# 'read' and 'write' halves of the increment -- that's what would happen on a
# real multi-machine system where network latency sits between the two steps.
import threading, time

def race_once(n_threads=10, per_thread=1_000):
    count = 0
    def worker_no_lock():
        nonlocal count
        for _ in range(per_thread):
            tmp = count          # read
            time.sleep(0)        # yield -- simulates network/GC pause mid-operation
            count = tmp + 1      # write (may overwrite another thread's update)
    ts = [threading.Thread(target=worker_no_lock) for _ in range(n_threads)]
    for t in ts: t.start()
    for t in ts: t.join()
    return count

EXPECTED = 10 * 1_000
results = [race_once() for _ in range(3)]
for i, c in enumerate(results, 1):
    print(f"run {i}: without lock -> count = {c:,} (expected {EXPECTED:,}) "
          f"-> lost {EXPECTED - c:,} updates")

# The bug is not a fluke: assert we really did lose updates every single time.
assert all(c < EXPECTED for c in results), "expected lost updates on every run"
print("\nEvery run lost updates. This is the failure a distributed lock exists to prevent.")

In [ ]:
# GOOD -- with a local mutex. Threads inside *one* process can coordinate via threading.Lock.
import threading, time

def guarded_once(n_threads=10, per_thread=1_000):
    count = 0
    mu = threading.Lock()
    def worker_with_lock():
        nonlocal count
        for _ in range(per_thread):
            with mu:
                tmp = count
                time.sleep(0)    # same yield as before -- the lock still protects us
                count = tmp + 1
    ts = [threading.Thread(target=worker_with_lock) for _ in range(n_threads)]
    for t in ts: t.start()
    for t in ts: t.join()
    return count

results = [guarded_once() for _ in range(3)]
print("with local lock ->", [f"{c:,}" for c in results])
assert all(c == 10_000 for c in results), "a correct lock must be exact every time"
print("Exact every time. Same load, same yields -- only the lock changed.")

A `threading.Lock` works because all threads share memory. But production workloads
run across **many machines** — the threads can't see each other's mutexes.

That's what a **distributed lock manager (DLM)** is: a separate service everyone
talks to, which plays the role of "the one true mutex" for the whole fleet.


## Back-of-the-envelope: how big does a lock service need to be?

A lock manager is one of the few services where the *interesting* number is not
storage — it is **write throughput** and **lease traffic**, because every lock
operation is a consensus write, and every live lock heartbeats forever.

Run the cell and change the inputs. The point is to notice which term dominates.

In [ ]:
# Sizing a distributed lock manager. Every number here is a write, not a read --
# that is what makes this service expensive compared to a cache.

def size_dlm(concurrent_locks: int,
             acquire_per_s: int,
             ttl_s: float,
             renew_fraction: float = 0.5,
             bytes_per_lock: int = 200,
             etcd_writes_per_s: int = 10_000):
    """
    concurrent_locks : how many locks are held at any instant
    acquire_per_s    : new acquire attempts per second (includes losers retrying)
    ttl_s            : lease length; holders renew at ttl_s * renew_fraction
    etcd_writes_per_s: ballpark sustained write ceiling of ONE Raft cluster
    """
    renew_interval = ttl_s * renew_fraction
    renew_qps   = concurrent_locks / renew_interval   # heartbeats never stop
    release_qps = acquire_per_s                       # every acquire eventually releases
    write_qps   = acquire_per_s + renew_qps + release_qps
    state_mb    = concurrent_locks * bytes_per_lock / 1e6
    clusters    = -(-write_qps // etcd_writes_per_s)  # ceiling division

    print(f"concurrent locks held : {concurrent_locks:>12,}")
    print(f"acquire QPS           : {acquire_per_s:>12,}")
    print(f"renew  QPS            : {renew_qps:>12,.0f}   (TTL {ttl_s}s, renew every {renew_interval}s)")
    print(f"release QPS           : {release_qps:>12,}")
    print(f"TOTAL consensus writes: {write_qps:>12,.0f} /s")
    print(f"state to keep in RAM  : {state_mb:>12,.1f} MB   <- tiny; this is never the problem")
    print(f"Raft clusters needed  : {clusters:>12,.0f}   (at ~{etcd_writes_per_s:,} writes/s each)")

print("=== 100k held locks, 10s TTL ===")
size_dlm(concurrent_locks=100_000, acquire_per_s=5_000, ttl_s=10)

print("\n=== same load, 60s TTL (6x fewer heartbeats) ===")
size_dlm(concurrent_locks=100_000, acquire_per_s=5_000, ttl_s=60)

### What the numbers say

- **Heartbeats dominate**, not acquires. 100k held locks on a 10s TTL cost 20,000
  renew-writes/second before a single new lock is taken. Stretching the TTL to 60s
  cuts total write load by roughly 3x.
- **State is trivial** — 100k locks is ~20 MB. A lock manager is never storage-bound.
- So the TTL is not just a safety knob, it is a **capacity knob**, and it pulls in
  two directions:

| TTL | Safety / liveness | Cost |
|---|---|---|
| Short (5–10s) | dead owner detected fast | heavy heartbeat write load |
| Long (60s+)   | cheap | a crashed holder blocks everyone for up to a minute |

- **One Raft cluster is a hard ceiling** (~10k sustained writes/s for etcd, and it
  does *not* scale by adding members — more members means slower commits). Past that
  you shard the lock **namespace** across independent clusters. Locks in different
  shards can no longer be taken atomically together, which is the real cost.

## Architecture options

### 1) Single Redis with `SET NX PX`
```
SET lock:name owner-A NX PX 30000
```
Simple, fast (sub-ms). But if the Redis master fails right after a `SET NX` and a replica
that doesn't have the write gets promoted → two clients can each hold the lock.

### 2) Redlock (multi-Redis)
Acquire a majority of M independent Redis instances. Widely debated —
[Kleppmann's critique](https://martin.kleppmann.com/2016/02/08/how-to-do-distributed-locking.html)
argues it's not safe under arbitrary clock skew. If you need correctness, prefer (3).

### 3) Zookeeper / etcd / Consul (consensus-based)
Each uses Raft/Paxos. A lock = ephemeral znode (ZK) or lease (etcd).
Slower per op (tens of ms), but *correct under partitions*. **Recommended** when
correctness matters (leader election, schema migrations, financial jobs).

### 4) Database row + TTL
```sql
UPDATE locks SET owner=?, expires=? WHERE name=? AND (owner IS NULL OR expires < now());
```
Works, boring, reuses infrastructure you already have. Pay DB latency per op.

### Picking one — quick rubric

| Need | Use |
|---|---|
| "I just want one cron to run" | Redis `SET NX PX` + fencing token |
| "Leader election for a stateful service" | etcd / ZooKeeper lease |
| "Can't add new infra" | Postgres row lock with `expires_at` |
| "Cross-region, partition-tolerant" | etcd / ZK (accept the latency) |


## Fencing tokens (why they matter)

```
         time ───────────────────────────────────▶

 Client A: acquire(lock=L, token=17) ────── GC pause ─── write(X)
                                                       ↑
                                                       still thinks it holds the lock!
 Client B:             acquire(lock=L, token=18) ── write(X)

 Resource (X):  must see the token and reject A's stale write
                because 17 < the last token (18) it saw from B.
```

The **lock service hands out a monotonically increasing token** on each acquire.
The **protected resource** (DB, file, queue, cache) must check
`incoming_token ≥ last_seen_token` on every operation.

Without this, TTL-based locks *cannot* guarantee mutual exclusion under pauses.
We'll build and break this end-to-end in Notebook 3.
